# CFO Estimation — Cross-Antenna Methods

Compares four carrier frequency offset (CFO) estimators for the preamble correlator.

All four SX1257 front-ends share one TCXO, so the CFO `df` is **identical** across all NR=4 antennas.
Only the channel phase `φ_j = ∠h_j` differs per antenna. This notebook quantifies how much
exploiting all four antennas jointly improves the CFO estimate.

## Methods compared

| # | Method | Antennas | SNR boost | Resolution |
|---|--------|----------|-----------|------------|
| 1 | Single antenna, integer-bin | 1 | baseline | BW/M |
| 2 | Single antenna, parabolic sub-bin | 1 | baseline | ~BW/10M |
| 3 | 4-antenna incoherent Σ\|FFT\|², sub-bin | 4 | 6 dB | ~BW/10M |
| 4 | 4-antenna coherent two-pass, sub-bin | 4 | up to 12 dB | ~BW/10M |

All methods operate on one preamble upchirp symbol (M samples per antenna). Multi-symbol
coherent integration (the existing 8-symbol correlator) is a separate, complementary gain.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('../..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sim.models.lora    import upchirp
from sim.models.channel import rayleigh_coefficients, apply_channel

rng = np.random.default_rng(42)
np.random.seed(42)

SF         = 9
M          = 2 ** SF      # 512 chips per symbol
BW         = 125e3        # Hz — chip rate equals bandwidth
Fs         = BW
NR         = 4            # receive antennas
BIN_HZ     = BW / M      # frequency resolution per FFT bin

print(f'SF={SF}  M={M}  BW={BW/1e3:.0f} kHz')
print(f'Bin width:           {BIN_HZ:.2f} Hz')
print(f'Integer-bin worst:  ±{BIN_HZ/2:.2f} Hz  (half bin)')
print(f'Dirichlet loss at half-bin: {20*np.log10(np.sinc(0.5)):.2f} dB')

## Helper functions

**Dechirp:** multiply received samples by `conj(upchirp)`. For a received upchirp with CFO `df`,
this leaves a pure complex tone at frequency `df`:
```
d_j[n] = rx_j[n] · c*[n] = h_j · exp(j·2π·df·n/Fs) + noise
```
Taking the FFT of `d_j` gives a peak at bin `k ≈ df·M/BW`.

**Parabolic sub-bin interpolation:** fits a parabola to the three bins around the
integer peak and finds the fractional-bin maximum:
```
k_frac = k_peak + 0.5·(α − γ) / (α − 2β + γ)    where α,β,γ = P[k±1], P[k]
```
Reduces the worst-case residual from ±BW/(2M) to roughly ±BW/(10M–20M).

In [ ]:
def dechirp(rx: np.ndarray) -> np.ndarray:
    """Remove chirp modulation from one received symbol."""
    n = np.arange(len(rx))
    return rx * np.exp(-1j * np.pi * n**2 / M)


def parabolic_subbin(P: np.ndarray, k: int) -> float:
    """Fractional bin location of the peak via 3-point parabola on spectrum P."""
    N   = len(P)
    a   = P[(k - 1) % N]
    b   = P[k]
    c   = P[(k + 1) % N]
    den = a - 2*b + c
    if abs(den) < 1e-12:
        return float(k)
    return k + 0.5 * (a - c) / den


def dirichlet_magnitude(df_residual: float) -> float:
    """Normalised FFT peak magnitude for a residual CFO of df_residual Hz."""
    arg  = np.pi * df_residual * M / Fs
    darg = np.pi * df_residual / Fs
    if abs(darg) < 1e-12:
        return 1.0
    return abs(np.sin(arg)) / (M * abs(np.sin(darg)))


def make_rx(df_true: float, h: np.ndarray, N0: float) -> np.ndarray:
    """Synthesise one received preamble upchirp across NR antennas with CFO and AWGN."""
    n  = np.arange(M)
    tx = upchirp(M) * np.exp(1j * 2 * np.pi * df_true * n / Fs)
    return np.array([apply_channel(tx, h[j], N0) for j in range(NR)])

## The four estimators

In [ ]:
def est_single_integer(rx: np.ndarray) -> float:
    """Method 1: antenna 0 only, integer-bin argmax."""
    D = np.fft.fft(dechirp(rx[0]))
    k = int(np.argmax(np.abs(D)))
    return k * BIN_HZ


def est_single_subbin(rx: np.ndarray) -> float:
    """Method 2: antenna 0 only, integer-bin + parabolic sub-bin."""
    D = np.fft.fft(dechirp(rx[0]))
    P = np.abs(D)
    k = int(np.argmax(P))
    return parabolic_subbin(P, k) * BIN_HZ


def est_incoherent_subbin(rx: np.ndarray) -> float:
    """Method 3: sum |FFT_j|² across all 4 antennas (incoherent), sub-bin.

    No prior knowledge of h_j needed. Gives 4× SNR improvement in the
    power spectrum over method 2, making sub-bin interpolation reliable
    further into the noise floor.
    """
    P_sum = np.zeros(M)
    for j in range(NR):
        D = np.fft.fft(dechirp(rx[j]))
        P_sum += np.abs(D) ** 2
    k = int(np.argmax(P_sum))
    return parabolic_subbin(P_sum, k) * BIN_HZ


def est_coherent_twopass(rx: np.ndarray) -> float:
    """Method 4: two-pass coherent cross-antenna combining, sub-bin.

    Pass 1: incoherent peak → coarse φ_j = ∠FFT_j[k_peak] per antenna.
    Pass 2: phase-align each antenna's dechirped samples by exp(-jφ_j),
            sum across antennas → single higher-SNR stream → FFT → sub-bin.

    Coherent combining gives up to NR× (6 dB) SNR advantage over method 3
    at high SNR, but degrades when pass-1 phase estimates are noisy.
    """
    # --- Pass 1 ---
    Ds    = [np.fft.fft(dechirp(rx[j])) for j in range(NR)]
    P_sum = sum(np.abs(D)**2 for D in Ds)
    k_peak = int(np.argmax(P_sum))
    phi_j  = np.array([np.angle(Ds[j][k_peak]) for j in range(NR)])

    # --- Pass 2 ---
    aligned = np.zeros(M, dtype=complex)
    for j in range(NR):
        aligned += dechirp(rx[j]) * np.exp(-1j * phi_j[j])

    P2 = np.abs(np.fft.fft(aligned))
    k2 = int(np.argmax(P2))
    return parabolic_subbin(P2, k2) * BIN_HZ


METHODS = {
    'Single ant, integer-bin':   (est_single_integer,    'C0', '--'),
    'Single ant, sub-bin':        (est_single_subbin,     'C1', '--'),
    '4-ant incoherent, sub-bin':  (est_incoherent_subbin, 'C2', '-'),
    '4-ant coherent 2-pass':      (est_coherent_twopass,  'C3', '-'),
}

## Single-trial visualisation

Apply a known CFO of 0.37 bins (worst-case is 0.5 bins) at moderate SNR.
The four panels show the power spectrum each estimator operates on, plus where it places its estimate.

In [ ]:
DF_DEMO  = 0.37 * BIN_HZ     # sub-bin offset — 0.5 is worst case
SNR_DEMO = 0.0               # dB per antenna
N0_DEMO  = 10 ** (-SNR_DEMO / 10)

h_demo   = rayleigh_coefficients(NR, pll_phase_random=True)
rx_demo  = make_rx(DF_DEMO, h_demo, N0_DEMO)

# Build the spectrum each method operates on
def spectrum_single(rx):  return np.abs(np.fft.fft(dechirp(rx[0])))
def spectrum_incoherent(rx):
    return sum(np.abs(np.fft.fft(dechirp(rx[j])))**2 for j in range(NR))
def spectrum_coherent_p2(rx):
    Ds     = [np.fft.fft(dechirp(rx[j])) for j in range(NR)]
    P_sum  = sum(np.abs(D)**2 for D in Ds)
    k_peak = int(np.argmax(P_sum))
    phi_j  = np.array([np.angle(Ds[j][k_peak]) for j in range(NR)])
    aligned = sum(dechirp(rx[j]) * np.exp(-1j*phi_j[j]) for j in range(NR))
    return np.abs(np.fft.fft(aligned))

spectra = [
    ('Method 1  Single ant, integer-bin',  spectrum_single(rx_demo),      est_single_integer(rx_demo)),
    ('Method 2  Single ant, sub-bin',       spectrum_single(rx_demo),      est_single_subbin(rx_demo)),
    ('Method 3  4-ant incoherent, sub-bin', spectrum_incoherent(rx_demo),  est_incoherent_subbin(rx_demo)),
    ('Method 4  4-ant coherent 2-pass',     spectrum_coherent_p2(rx_demo), est_coherent_twopass(rx_demo)),
]

# Zoom window: ±4 bins around true peak
k_true  = DF_DEMO / BIN_HZ
k_centre = round(k_true)
k_lo, k_hi = k_centre - 4, k_centre + 5
k_axis  = np.arange(k_lo, k_hi)
freq_axis = k_axis * BIN_HZ

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=False)
fig.suptitle(f'Single-trial spectra — CFO = {DF_DEMO:.1f} Hz ({DF_DEMO/BIN_HZ:.2f} bins),  SNR = {SNR_DEMO} dB/ant',
             fontsize=11)

for ax, (title, P, df_est) in zip(axes, spectra):
    p_slice = P[k_axis % M]
    ax.bar(freq_axis, p_slice / p_slice.max(), width=BIN_HZ * 0.8,
           color='C0', alpha=0.6, label='spectrum')
    ax.axvline(DF_DEMO,  color='green', lw=1.5, linestyle='--', label=f'true {DF_DEMO:.1f} Hz')
    ax.axvline(df_est,   color='red',   lw=1.5, linestyle=':',  label=f'est  {df_est:.1f} Hz')
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Frequency (Hz)')
    ax.legend(fontsize=7)
    err = df_est - DF_DEMO
    ax.set_xlabel(f'residual = {err:+.2f} Hz', fontsize=8)

axes[0].set_ylabel('Normalised magnitude')
plt.tight_layout()
plt.show()

## Residual error shape — sweep df across one bin

Sweep the true CFO from 0 to BW/M (one bin width) at high SNR. This isolates the
sub-bin interpolation error from the bin-detection failure that dominates at low SNR.
Each method's residual should be symmetric around the integer-bin centres.

In [ ]:
N_SWEEP   = 200
N0_HIGH   = 10**(-20/10)    # 20 dB SNR — peak detection essentially perfect
df_sweep  = np.linspace(0, BIN_HZ, N_SWEEP, endpoint=False)
N_AVG     = 50              # trials per df point (for noise averaging)

sweep_results = {name: np.zeros(N_SWEEP) for name in METHODS}

for i, df_true in enumerate(df_sweep):
    for name, (fn, _, _) in METHODS.items():
        errs = []
        for _ in range(N_AVG):
            h   = rayleigh_coefficients(NR, pll_phase_random=True)
            rx  = make_rx(df_true, h, N0_HIGH)
            df_est = fn(rx)
            errs.append(((df_est - df_true + BW/2) % BW) - BW/2)
        sweep_results[name][i] = np.mean(errs)

fig, ax = plt.subplots(figsize=(10, 4))
for name, (_, col, ls) in METHODS.items():
    ax.plot(df_sweep / BIN_HZ, sweep_results[name], color=col, linestyle=ls,
            lw=1.5, label=name)

ax.axhline(0, color='k', lw=0.5)
ax.axhline( BIN_HZ/2,  color='grey', lw=0.6, linestyle=':', label='±half-bin')
ax.axhline(-BIN_HZ/2,  color='grey', lw=0.6, linestyle=':')
ax.set_xlabel('True CFO (bins)')
ax.set_ylabel('Mean residual error (Hz)')
ax.set_title(f'Residual CFO error vs sub-bin offset  (SNR=20 dB, averaged over {N_AVG} channel realisations)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print('Worst-case |residual| (Hz):')
for name in METHODS:
    wc = np.max(np.abs(sweep_results[name]))
    print(f'  {name:38s}  {wc:6.2f} Hz  ({wc/BIN_HZ*100:.1f}% of bin)')

## Monte Carlo — RMS residual CFO error vs SNR

True CFO drawn uniformly from `[0, BW)` so the result includes both sub-bin error
(high SNR regime) and bin-detection failures (low SNR regime where the argmax lands
on the wrong bin and the residual jumps by ~BW/M).

In [ ]:
N_TRIALS    = 1000
SNR_DB_AXIS = np.arange(-20, 11, 2)   # dB per antenna

mc_rms   = {name: [] for name in METHODS}   # RMS residual (Hz)
mc_p_bad = {name: [] for name in METHODS}   # fraction of trials with |residual| > BIN_HZ/2

for snr_db in SNR_DB_AXIS:
    N0 = 10**(-snr_db / 10)
    sq_errs  = {name: [] for name in METHODS}
    bad_flag = {name: [] for name in METHODS}

    for _ in range(N_TRIALS):
        df_true = rng.uniform(0, BW)
        h       = rayleigh_coefficients(NR, pll_phase_random=True)
        rx      = make_rx(df_true, h, N0)

        for name, (fn, _, _) in METHODS.items():
            df_est   = fn(rx)
            residual = ((df_est - df_true + BW/2) % BW) - BW/2
            sq_errs[name].append(residual**2)
            bad_flag[name].append(abs(residual) > BIN_HZ / 2)

    for name in METHODS:
        mc_rms[name].append(np.sqrt(np.mean(sq_errs[name])))
        mc_p_bad[name].append(np.mean(bad_flag[name]))

print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Left: RMS residual error ---
ax = axes[0]
for name, (_, col, ls) in METHODS.items():
    ax.semilogy(SNR_DB_AXIS, mc_rms[name], color=col, linestyle=ls, lw=2, marker='o',
                ms=4, label=name)

ax.axhline(BIN_HZ / 2,  color='grey', lw=0.8, linestyle=':', label='±half-bin (integer floor)')
ax.axhline(BIN_HZ / 20, color='grey', lw=0.8, linestyle='--', label='±1/20 bin (sub-bin target)')
ax.set_xlabel('Per-antenna SNR (dB)')
ax.set_ylabel('RMS residual CFO error (Hz)')
ax.set_title('RMS CFO estimation error vs SNR')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.2)
ax.set_ylim(1e0, BW)

# --- Right: bin-detection failure rate ---
ax = axes[1]
for name, (_, col, ls) in METHODS.items():
    ax.semilogy(SNR_DB_AXIS, np.clip(mc_p_bad[name], 1e-4, 1),
                color=col, linestyle=ls, lw=2, marker='o', ms=4, label=name)

ax.set_xlabel('Per-antenna SNR (dB)')
ax.set_ylabel('P(|residual| > half-bin)')
ax.set_title('Bin-detection failure rate')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.2)

plt.suptitle(f'SF={SF}  BW={BW/1e3:.0f} kHz  NR={NR}  N={N_TRIALS} trials', fontsize=10)
plt.tight_layout()
plt.show()

## FFT peak loss from residual CFO

The residual CFO after estimation attenuates the dechirped FFT peak by the Dirichlet sinc factor:
```
gain(Δf) = |sin(π·Δf·M/Fs)| / (M·|sin(π·Δf/Fs)|)  ≈  sinc(Δf·M/Fs)
```
This translates directly to sensitivity loss at the SX1302 demodulator.

In [ ]:
# For each method at each SNR, compute the mean and worst-case Dirichlet loss
# from the actual residuals (rerun a smaller MC for this)

N_LOSS = 500
mc_loss_db = {name: [] for name in METHODS}

for snr_db in SNR_DB_AXIS:
    N0 = 10**(-snr_db / 10)
    losses = {name: [] for name in METHODS}

    for _ in range(N_LOSS):
        # Draw df within a single bin so bin-detection failures don't dominate
        # the loss stat (those are captured by the failure-rate plot already)
        df_true = rng.uniform(0, BIN_HZ)
        h       = rayleigh_coefficients(NR, pll_phase_random=True)
        rx      = make_rx(df_true, h, N0)

        for name, (fn, _, _) in METHODS.items():
            df_est   = fn(rx)
            residual = ((df_est - df_true + BW/2) % BW) - BW/2
            gain     = dirichlet_magnitude(residual)
            losses[name].append(20 * np.log10(max(gain, 1e-6)))

    for name in METHODS:
        mc_loss_db[name].append(np.mean(losses[name]))

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (_, col, ls) in METHODS.items():
    ax.plot(SNR_DB_AXIS, mc_loss_db[name], color=col, linestyle=ls, lw=2, marker='o',
            ms=4, label=name)

ax.axhline(-3.9, color='grey', lw=0.8, linestyle=':', label='−3.9 dB (half-bin worst case)')
ax.axhline(-0.1, color='grey', lw=0.8, linestyle='--', label='−0.1 dB (sub-bin target)')
ax.set_xlabel('Per-antenna SNR (dB)')
ax.set_ylabel('Mean FFT peak loss (dB)')
ax.set_title(f'Dirichlet peak loss from residual CFO  (df drawn within one bin, SF={SF}  BW={BW/1e3:.0f} kHz)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)
ax.set_ylim(-5, 0.5)
plt.tight_layout()
plt.show()

print(f'\nAt SNR = -5 dB:')
idx = list(SNR_DB_AXIS).index(-5) if -5 in SNR_DB_AXIS else np.argmin(np.abs(SNR_DB_AXIS - (-5)))
for name in METHODS:
    print(f'  {name:38s}  {mc_loss_db[name][idx]:+.2f} dB')

## Coherent vs incoherent — where does the threshold fall?

Method 4 (coherent two-pass) uses per-antenna phase estimates `φ_j` from pass 1 to align
the antennas before summing. At low SNR these phase estimates are noisy, which degrades
the coherent combining gain. There is a threshold SNR below which method 4 offers no
advantage over the simpler method 3.

This cell isolates that crossover by fixing `df` at the worst-case half-bin offset and
measuring the RMS residual as a function of SNR.

In [ ]:
N_THRESHOLD = 1000
DF_WORST    = BIN_HZ / 2      # maximum sub-bin error for integer-bin methods

thresh_rms = {name: [] for name in METHODS}

for snr_db in SNR_DB_AXIS:
    N0 = 10**(-snr_db / 10)
    errs = {name: [] for name in METHODS}

    for _ in range(N_THRESHOLD):
        h  = rayleigh_coefficients(NR, pll_phase_random=True)
        rx = make_rx(DF_WORST, h, N0)

        for name, (fn, _, _) in METHODS.items():
            df_est   = fn(rx)
            residual = ((df_est - DF_WORST + BW/2) % BW) - BW/2
            errs[name].append(residual**2)

    for name in METHODS:
        thresh_rms[name].append(np.sqrt(np.mean(errs[name])))

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, (_, col, ls) in METHODS.items():
    ax.semilogy(SNR_DB_AXIS, thresh_rms[name], color=col, linestyle=ls, lw=2,
                marker='o', ms=4, label=name)

ax.axhline(BIN_HZ / 2, color='grey', lw=0.8, linestyle=':', label='half-bin = input df')
ax.set_xlabel('Per-antenna SNR (dB)')
ax.set_ylabel('RMS residual CFO error (Hz)')
ax.set_title(f'Worst-case df = {DF_WORST:.1f} Hz (half-bin)\nCoherent vs incoherent threshold')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

# Find crossover SNR between methods 3 and 4
rms3 = np.array(thresh_rms['4-ant incoherent, sub-bin'])
rms4 = np.array(thresh_rms['4-ant coherent 2-pass'])
crossings = np.where(np.diff(np.sign(rms3 - rms4)))[0]
if len(crossings):
    snr_cross = SNR_DB_AXIS[crossings[0]]
    print(f'Methods 3 vs 4 crossover: ~{snr_cross} dB per-antenna SNR')
    print('Below this SNR, incoherent (method 3) is as good or better than coherent (method 4).')
else:
    print('No crossover found in the swept range — check if coherent is always better or always worse.')

## Correlator bandwidth vs residual CFO

Using more preamble symbols for the correlator gives more integration gain (+3 dB per doubling)
but **narrows its frequency mainlobe**, making it more sensitive to residual CFO from the
estimation step. The first null of an N_sym-symbol correlator is at:
```
null = BW / (N_sym × M)
```
The net channel estimation quality (h_hat SNR relative to a 1-symbol corrected reference) is:
```
Q(N_corr, δf) ∝ N_corr × |dirichlet(δf, N_corr·M)|²
```
where δf is the residual CFO after the estimation step. For each pair of N_corr values
there is a **crossover residual** δf* below which the longer correlator wins and above which
the shorter one wins — because narrower mainlobe loss outweighs the extra integration gain.

The vertical marker lines show worst-case residuals for each CFO estimation strategy:
- **1-symbol FFT, integer-bin:** ±BW/(2M) = ±122 Hz
- **1-symbol FFT, sub-bin:** ~19% of bin ≈ ±46 Hz (from sweep above)
- **8-symbol FFT, integer-bin:** ±BW/(16M) = ±15 Hz
- **8-symbol FFT, sub-bin:** ~25% of 8-sym bin ≈ ±4 Hz (estimated)

In [ ]:
from scipy.optimize import brentq

BIN_HZ_8SYM = BW / (8 * M)   # 8-symbol FFT bin width (30.5 Hz at SF9)

def dirichlet_corr(df_res: float, N_sym: int) -> float:
    """Dirichlet magnitude of N_sym-symbol correlator at residual CFO df_res Hz."""
    N    = N_sym * M
    arg  = np.pi * df_res * N / Fs
    darg = np.pi * df_res / Fs
    if abs(darg) < 1e-12:
        return 1.0
    return abs(np.sin(arg)) / (N * abs(np.sin(darg)))

def Q_db(df_res: float, N_corr: int) -> float:
    """Net h_hat SNR gain vs 1-symbol corrected reference (dB)."""
    return 10 * np.log10(N_corr * dirichlet_corr(df_res, N_corr) ** 2)

df_axis       = np.linspace(0.01, BIN_HZ / 2, 800)   # 0 to 122 Hz (half 1-sym bin)
N_CORR_OPTS   = [2, 4, 8]
CORR_COLS     = ['C4', 'C5', 'C6']
CORR_LABELS   = [f'N_corr = {n}  (null at {BW/(n*M):.0f} Hz)' for n in N_CORR_OPTS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: net h_hat SNR vs residual CFO ---
ax = axes[0]
for nc, col, lbl in zip(N_CORR_OPTS, CORR_COLS, CORR_LABELS):
    ax.plot(df_axis, [Q_db(df, nc) for df in df_axis], color=col, lw=2, label=lbl)

residual_marks = [
    (BIN_HZ / 2,         'tomato',      ':',  f'1-sym int-bin  ±{BIN_HZ/2:.0f} Hz'),
    (BIN_HZ * 0.19,      'tomato',      '--', f'1-sym sub-bin  ±{BIN_HZ*0.19:.0f} Hz'),
    (BIN_HZ_8SYM,        'mediumblue',  ':',  f'8-sym int-bin  ±{BIN_HZ_8SYM:.1f} Hz'),
    (BIN_HZ_8SYM * 0.25, 'mediumblue',  '--', f'8-sym sub-bin  ±{BIN_HZ_8SYM*0.25:.1f} Hz'),
]
for df_m, col, ls, lbl in residual_marks:
    ax.axvline(df_m, color=col, lw=1.1, linestyle=ls, label=lbl)

ax.set_xlabel('Residual CFO δf after estimation (Hz)')
ax.set_ylabel('Net h_hat SNR gain vs 1-sym reference (dB)')
ax.set_title('Correlator net SNR: integration gain minus CFO loss')
ax.legend(fontsize=7.5, ncol=2)
ax.grid(True, alpha=0.2)

# --- Right: crossover points ---
ax = axes[1]
for nc, col, lbl in zip(N_CORR_OPTS, CORR_COLS, CORR_LABELS):
    ax.plot(df_axis, [Q_db(df, nc) for df in df_axis], color=col, lw=2, label=lbl)

pairs       = [(8, 4), (8, 2), (4, 2)]
cross_cols  = ['red', 'darkorange', 'purple']
crossovers  = []
for (n1, n2), ccol in zip(pairs, cross_cols):
    try:
        # Search only up to just below the first null of the longer (n1) correlator,
        # where Q_db(n1) - Q_db(n2) is monotone — avoids brentq sign-change failure.
        ub = 0.99 * BW / (n1 * M)
        df_c = brentq(lambda df: Q_db(df, n1) - Q_db(df, n2), 0.1, ub)
        crossovers.append((n1, n2, df_c))
        ax.axvline(df_c, color=ccol, lw=1.2, linestyle='-.',
                   label=f'N={n1}↔N={n2}  at {df_c:.1f} Hz')
        ax.annotate(f'{df_c:.1f} Hz', xy=(df_c, Q_db(df_c, n1) + 0.1),
                    xytext=(df_c + 2, Q_db(df_c, n1) + 0.3),
                    fontsize=7.5, color=ccol,
                    arrowprops=dict(arrowstyle='->', color=ccol, lw=0.8))
    except ValueError:
        pass

# shade the 8-symbol FFT integer-bin zone
ax.axvspan(0, BIN_HZ_8SYM, alpha=0.10, color='mediumblue',
           label=f'8-sym FFT zone (δf < {BIN_HZ_8SYM:.0f} Hz)')

ax.set_xlabel('Residual CFO δf after estimation (Hz)')
ax.set_ylabel('Net h_hat SNR gain (dB)')
ax.set_title('Crossover residuals — optimal N_corr by CFO quality')
ax.legend(fontsize=7.5)
ax.grid(True, alpha=0.2)

plt.suptitle(f'SF={SF}  BW={BW/1e3:.0f} kHz  M={M}  1-sym bin = {BIN_HZ:.1f} Hz   8-sym bin = {BIN_HZ_8SYM:.1f} Hz',
             fontsize=10)
plt.tight_layout()
plt.show()

print(f'Correlator first-null frequencies:')
for nc in N_CORR_OPTS:
    print(f'  N_corr={nc}: null at {BW/(nc*M):.1f} Hz')

print(f'\nCrossover residual CFOs (above → use fewer symbols):')
for n1, n2, df_c in crossovers:
    print(f'  N={n1} vs N={n2}: {df_c:.2f} Hz')

print(f'\nOptimal N_corr for each CFO estimation strategy:')
scenarios = [
    ('1-sym FFT, integer-bin',  BIN_HZ / 2),
    ('1-sym FFT, sub-bin',      BIN_HZ * 0.19),
    ('8-sym FFT, integer-bin',  BIN_HZ_8SYM),
    ('8-sym FFT, sub-bin',      BIN_HZ_8SYM * 0.25),
]
for label, df_wc in scenarios:
    best = max(N_CORR_OPTS, key=lambda n: Q_db(df_wc, n))
    gains = {n: Q_db(df_wc, n) for n in N_CORR_OPTS}
    print(f'  {label:30s} (δf_wc={df_wc:.1f} Hz)  → N_corr={best}',
          f'  ({gains[8]:+.2f} / {gains[4]:+.2f} / {gains[2]:+.2f} dB for N=8/4/2)')

## Summary

### CFO estimation method

| Method | High-SNR residual | Low-SNR benefit | Hardware cost |
|--------|------------------|-----------------|---------------|
| 1 — single ant, integer | ±122 Hz (SF9) | none | none |
| 2 — single ant, sub-bin | ~±46 Hz | none | 3-point parabola in FW |
| 3 — 4-ant incoherent, sub-bin | ~±46 Hz | 6 dB lower threshold | sum 4 FFTs (already available) |
| 4 — 4-ant coherent 2-pass | ~±46 Hz | up to 12 dB, degrades below crossover | 2nd FFT pass + phase rotation |

**Recommended estimator:** Method 3 (incoherent Σ|FFT|²) — no prior h_j needed,
no second pass, fits existing hardware.

### Correlator symbol count

More correlator symbols → +3 dB per doubling but narrower mainlobe (first null = BW/(N·M)).
The crossover residual CFO at which N_corr=8 and N_corr=4 are equal is **~13 Hz**.

| CFO estimation | Worst-case residual | Optimal N_corr | Rationale |
|----------------|--------------------|-----------------|-----------|
| 1-sym FFT, integer-bin | ±122 Hz | **2** | residual >> crossover |
| 1-sym FFT, sub-bin | ~±46 Hz | **2–4** | residual near N=4 vs N=2 crossover (~26 Hz) |
| 8-sym FFT, integer-bin | ±15 Hz | **4** | residual above N=8 vs N=4 crossover (13 Hz) |
| 8-sym FFT, sub-bin | ~±4 Hz | **8** | residual well below crossover |

**Key result:** sub-bin interpolation is not optional — it is the step that unlocks the full
8-symbol correlator. Without it, the 8-symbol correlator's −3.9 dB CFO loss exactly cancels
its +3 dB integration gain, making it no better than a 4-symbol correlator.

### Recommended pipeline

```
8-sym dechirp FFT + 4-ant incoherent Σ|FFT|² + parabolic sub-bin
    → δf_est  (residual ~±4 Hz)
    → apply exp(-j2π·df_est·n/Fs) correction to all 8 symbols in capture SRAM
    → 8-symbol correlator on corrected samples
    → h_hat with full integration gain and <0.1 dB CFO loss
```

**Architecture note:** no additional SRAM required. Raw preamble samples remain in
capture region (`0x08000+`); FFT staging (`0x00000–0x07FFF`) is reused for both passes.
Timing overhead ~3.9 ms at SF12, vs 139 ms available window — 36× margin.